# Stage 4 — Downstream Detection & Action Evaluation

This stage takes the Stage 3 selective temporal token-reuse prototype and tests whether its reconstructed DINOv2 representations remain useful for the professor's downstream perception heads.

## Goal

Compare:

1. **Baseline:** full DINOv2 → detection + action
2. **Temporal reuse:** previous-frame cache → changed-token transformer recomputation → reconstructed DINOv2 features → the same detection + action heads

The baseline is treated as the reference output.

## Important scientific distinction

Stage 3 showed that about 74.8% of tokens were reusable with the selected lightweight image-space detector.

Stage 4 asks the more important question:

> **Does using those reused/reconstructed features preserve downstream perception outputs?**

This notebook does **not** claim that token reuse is lossless. DINOv2 uses global self-attention, so the reduced-token path is an approximation.

### Required Kaggle inputs

- WoodScape dataset
- The professor's `UnifiedPerception` package / training package

The Stage 3 CSV/ZIP is **not required**.

In [ ]:
from pathlib import Path
import os, sys, re, json, time, shutil, zipfile, warnings
warnings.filterwarnings("ignore")

DATA_ROOT = Path("/kaggle/input/datasets/saanvishetty0804/car-nvidia")
CURRENT_DIR = DATA_ROOT / "rgb_images-001" / "rgb_images"
PREVIOUS_DIR = DATA_ROOT / "previous_images-001" / "previous_images"

assert CURRENT_DIR.exists(), f"Missing {CURRENT_DIR}"
assert PREVIOUS_DIR.exists(), f"Missing {PREVIOUS_DIR}"

OUT = Path("/kaggle/working/stage4_downstream")
OUT.mkdir(parents=True, exist_ok=True)

print("WoodScape:", DATA_ROOT)
print("Output:", OUT)

In [ ]:
import torch, numpy as np, pandas as pd
from PIL import Image
import torch.nn.functional as F
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "Enable a Kaggle GPU."
DEVICE = torch.device("cuda")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## Configuration

Use the same configuration as the validated Stage 3 run:

- DINOv2 ViT-S/14
- 640×640
- 45×45 patch grid
- 2025 patch tokens + CLS
- 384-dimensional embeddings
- 32 pairs per camera
- image-space patch difference threshold = 0.08
- candidate Stage 2 cosine reference = 0.95

The first frame of each camera establishes the cache; downstream comparison is performed on the subsequent temporal-reuse frames.

In [ ]:
MODEL_ID = "facebook/dinov2-small"
IMAGE_SIZE = 640
PATCH_SIZE = 14
GRID = 45
NUM_PATCHES = GRID * GRID
DIM = 384
CAMERAS = ["FV", "RV", "MVL", "MVR"]

PAIRS_PER_CAMERA = 32
PIXEL_DIFF_THRESHOLD = 0.08
COSINE_REFERENCE = 0.95

print("Model:", MODEL_ID)
print("Pairs/camera:", PAIRS_PER_CAMERA)
print("Patch tokens:", NUM_PATCHES)
print("Pixel threshold:", PIXEL_DIFF_THRESHOLD)

## 1. Build verified current/previous pairs

WoodScape provides current images and corresponding previous images. Pairing is done using `(camera, frame_id)`, not directory order.

In [ ]:
def parse_name(path):
    m = re.match(r"^(\d+)_([A-Za-z0-9]+)(?:_prev)?\.png$", path.name)
    if not m:
        return None, None
    return int(m.group(1)), m.group(2)

previous_lookup = {}
for p in PREVIOUS_DIR.glob("*.png"):
    fid, cam = parse_name(p)
    if fid is not None and cam in CAMERAS:
        previous_lookup[(cam, fid)] = p

pairs = {c: [] for c in CAMERAS}
for p in CURRENT_DIR.glob("*.png"):
    fid, cam = parse_name(p)
    if fid is None or cam not in CAMERAS:
        continue
    prev = previous_lookup.get((cam, fid))
    if prev is not None:
        pairs[cam].append({
            "camera": cam,
            "frame_id": fid,
            "current": p,
            "previous": prev,
        })

for c in CAMERAS:
    pairs[c].sort(key=lambda x: x["frame_id"])
    print(c, len(pairs[c]), "available pairs")

assert all(len(pairs[c]) >= PAIRS_PER_CAMERA for c in CAMERAS)

for c in CAMERAS:
    print(c, "selected frame IDs:", [x["frame_id"] for x in pairs[c][:PAIRS_PER_CAMERA]])

## 2. Load DINOv2 directly

The backbone is loaded directly from Hugging Face so the same token-level implementation used in Stage 3 can be evaluated.

For the downstream heads, we also need the professor's `UnifiedPerception` package.

In [ ]:
HF_CACHE = Path("/kaggle/working/huggingface_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)

from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
    use_fast=False,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
    output_hidden_states=True,
).to(DEVICE).eval()

for p in model.parameters():
    p.requires_grad = False

print("DINOv2 loaded.")

In [ ]:
# Locate the professor's UnifiedPerception package in Kaggle Input.
candidates = []

for p in Path("/kaggle/input").rglob("UnifiedPerception"):
    if (p / "unified_perception").is_dir() and (p / "third_party" / "training_package").is_dir():
        candidates.append(p)

if not candidates:
    for p in Path("/kaggle/input").rglob("UnifiedPerception.zip"):
        candidates.append(p)

print("UnifiedPerception candidates:")
for p in candidates[:10]:
    print(" ", p)

assert candidates, (
    "UnifiedPerception was not found under /kaggle/input. "
    "Attach the professor's UnifiedPerception dataset/package."
)

UP_SRC = candidates[0]
UP_WORK = Path("/kaggle/working/UnifiedPerception_stage4")

if UP_WORK.exists():
    shutil.rmtree(UP_WORK)

if UP_SRC.is_file() and UP_SRC.suffix.lower() == ".zip":
    UP_WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UP_SRC) as z:
        z.extractall(UP_WORK)

    # Normalize one extra top-level folder if the ZIP contains UnifiedPerception/
    nested = UP_WORK / "UnifiedPerception"
    if nested.is_dir() and (nested / "unified_perception").is_dir():
        for child in nested.iterdir():
            shutil.move(str(child), str(UP_WORK / child.name))
        nested.rmdir()
else:
    shutil.copytree(UP_SRC, UP_WORK)

print("Working UnifiedPerception:", UP_WORK)

## 3. Patch the validated shared detection path if needed

The professor's shared detection implementation requires the YOLO detection head to expose `end2end=True` for the vendored YOLO26 path.

This cell patches the **working copy only**. The Kaggle input is not modified.

In [ ]:
det_file = UP_WORK / "unified_perception" / "heads" / "detection.py"
assert det_file.exists(), det_file

txt = det_file.read_text()

needle = "self.detect_head.training = False"
replacement = (
    "self.detect_head.training = False\n"
    "                self.detect_head.end2end = True"
)

if "self.detect_head.end2end = True" not in txt:
    assert needle in txt, "Could not find the expected YOLO26 detection-head location."
    txt = txt.replace(needle, replacement, 1)
    det_file.write_text(txt)
    print("Applied YOLO26 end2end patch.")
else:
    print("YOLO26 end2end patch already present.")

sys.path.insert(0, str(UP_WORK))

In [ ]:
from unified_perception.perception_types import BackboneOutput
from unified_perception.heads.detection import DetectionHead
from unified_perception.heads.action import ActionHead

TRAINING_PACKAGE = UP_WORK / "third_party" / "training_package"

detection_head = DetectionHead(
    training_package=str(TRAINING_PACKAGE),
    adapter_type="vit",
    conf_thres=0.25,
    mode="shared",
)

action_head = ActionHead(
    head_ckpt=str(UP_WORK / "assets" / "k400_outputs" / "best.pth"),
    classes_txt=str(UP_WORK / "assets" / "kinetics400" / "annotations" / "classes.txt"),
    topk=5,
    device="cuda",
)

print("Detection head:", detection_head.mode)
print("Action head loaded lazily.")

## 4. Image preprocessing

DINOv2 receives ImageNet-normalized tensors.

The detection adapter's detail branch expects the raw `[0,1]` image tensor, so both versions are retained.

In [ ]:
def load_image_tensors(path):
    image = Image.open(path).convert("RGB").resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        Image.Resampling.BICUBIC,
    )

    raw = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0)
    raw = raw.permute(2, 0, 1).unsqueeze(0).contiguous().to(DEVICE)

    norm = processor(
        images=image,
        return_tensors="pt",
        do_resize=False,
        do_center_crop=False,
    )["pixel_values"].to(DEVICE)

    assert tuple(raw.shape) == (1, 3, IMAGE_SIZE, IMAGE_SIZE)
    assert tuple(norm.shape) == (1, 3, IMAGE_SIZE, IMAGE_SIZE)
    return raw, norm

In [ ]:
@torch.inference_mode()
def full_forward(path):
    raw, norm = load_image_tensors(path)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model(pixel_values=norm, output_hidden_states=True)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return out, raw, elapsed

smoke, smoke_raw, smoke_time = full_forward(pairs["FV"][0]["current"])
assert tuple(smoke.last_hidden_state.shape) == (1, 2026, 384)
assert len(smoke.hidden_states) == 13

print("Smoke test PASSED")
print("Runtime:", round(smoke_time, 4), "s")
print("Tokens:", tuple(smoke.last_hidden_state.shape))
print("Hidden states:", len(smoke.hidden_states))

## 5. Lightweight image-space change detector

This is identical in principle to Stage 3: mean absolute RGB difference within each 14×14 patch.

A lower threshold selects more tokens; the final Stage 3 value is 0.08.

In [ ]:
def image_patch_change_mask(current_path, previous_path, threshold=PIXEL_DIFF_THRESHOLD):
    a = np.asarray(
        Image.open(current_path).convert("RGB").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BICUBIC
        ),
        dtype=np.float32,
    ) / 255.0

    b = np.asarray(
        Image.open(previous_path).convert("RGB").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BICUBIC
        ),
        dtype=np.float32,
    ) / 255.0

    diff = np.abs(a - b).mean(axis=2)
    diff = diff[:GRID * PATCH_SIZE, :GRID * PATCH_SIZE]

    patch_diff = diff.reshape(
        GRID, PATCH_SIZE, GRID, PATCH_SIZE
    ).mean(axis=(1, 3))

    mask = patch_diff >= threshold
    return mask.reshape(-1), patch_diff

## 6. Build a layer-wise previous-frame cache

Unlike the Stage 3 reporting notebook, Stage 4 reconstructs **all 13 hidden states** because the professor's detection adapter uses intermediate DINOv2 layers `[3, 6, 9, 12]`.

The cache therefore contains:

- hidden state 0: embeddings
- hidden states 1–12: transformer layer outputs

The final DINOv2 layer normalization is applied separately for the selective CLS representation.

In [ ]:
layers = model.encoder.layer if hasattr(model.encoder, "layer") else model.encoder.layers
assert len(layers) == 12

def run_layer(layer, hidden_states):
    try:
        result = layer(hidden_states=hidden_states)
    except TypeError:
        result = layer(hidden_states)

    if isinstance(result, tuple):
        return result[0]
    if hasattr(result, "last_hidden_state"):
        return result.last_hidden_state
    if hasattr(result, "hidden_states"):
        return result.hidden_states
    return result

@torch.inference_mode()
def build_previous_cache(path):
    _, norm = load_image_tensors(path)
    emb = model.embeddings(norm)
    cache = [emb.detach()]

    h = emb
    for layer in layers:
        h = run_layer(layer, h)
        cache.append(h.detach())

    assert len(cache) == 13
    return cache

print("Transformer layers:", len(layers))

## 7. Layer-wise selective recomputation

For each transformer layer:

1. keep the previous hidden state at stable positions,
2. replace selected positions with current-frame representations,
3. run attention only on CLS + changed patch tokens,
4. scatter those outputs back into the full token grid.

This is intentionally an experimental approximation to global attention.

In [ ]:
@torch.inference_mode()
def selective_forward(current_path, previous_cache, changed_mask):
    raw, norm = load_image_tensors(current_path)

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    current_emb = model.embeddings(norm)

    changed_idx = torch.nonzero(
        torch.as_tensor(changed_mask, device=DEVICE),
        as_tuple=False,
    ).flatten() + 1

    selected_idx = torch.cat([
        torch.zeros(1, dtype=torch.long, device=DEVICE),
        changed_idx,
    ])

    # Layer 0 / embedding state.
    reconstructed_cache = [previous_cache[0].clone()]
    reconstructed_cache[0][:, selected_idx, :] = current_emb[:, selected_idx, :]

    h_selected = current_emb[:, selected_idx, :]

    for layer_idx, layer in enumerate(layers, start=1):
        h_selected = run_layer(layer, h_selected)

        reconstructed = previous_cache[layer_idx].clone()
        reconstructed[:, selected_idx, :] = h_selected
        reconstructed_cache.append(reconstructed)

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    final_norm = model.layernorm(reconstructed_cache[-1])

    return {
        "hidden_states": reconstructed_cache,
        "last_hidden": final_norm,
        "raw": raw,
        "selected_tokens": int(len(selected_idx)),
        "elapsed": elapsed,
    }

## 8. Detection comparison metrics

The baseline detector is treated as the reference.

For each frame we report:

- baseline detection count
- selective detection count
- baseline detections matched by selective detections
- selective detections matched by baseline detections
- mean IoU of matched detections
- confidence difference on matched detections

Matching requires the same class and IoU ≥ 0.50.

In [ ]:
def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0

def compare_detections(base, sel, iou_threshold=0.50):
    used = set()
    matches = []

    for b in base:
        best = None
        best_iou = 0.0

        for j, s in enumerate(sel):
            if j in used or b["class_id"] != s["class_id"]:
                continue

            iou = box_iou(b["bbox"], s["bbox"])
            if iou >= iou_threshold and iou > best_iou:
                best_iou = iou
                best = j

        if best is not None:
            used.add(best)
            matches.append((b, sel[best], best_iou))

    n_base = len(base)
    n_sel = len(sel)

    return {
        "baseline_dets": n_base,
        "selective_dets": n_sel,
        "matched_dets": len(matches),
        "baseline_match_rate": len(matches) / n_base if n_base else 1.0,
        "selective_match_rate": len(matches) / n_sel if n_sel else 1.0,
        "mean_matched_iou": (
            float(np.mean([x[2] for x in matches])) if matches else 0.0
        ),
        "mean_conf_abs_diff": (
            float(np.mean([abs(x[0]["conf"] - x[1]["conf"]) for x in matches]))
            if matches else 0.0
        ),
    }

## 9. Run downstream evaluation

For each of the 4 cameras:

- 32 current/previous pairs are selected.
- The first frame is the cache initialization frame.
- For each subsequent frame:
  - full DINOv2 is computed,
  - previous-frame DINOv2 cache is computed from the paired previous image,
  - selective features are reconstructed,
  - both feature sets are passed through the same detection head,
  - CLS features are stored for action recognition.

The full previous-frame cache is **not counted as deployment latency** because in a streaming system it is already available from the preceding accepted frame. It is computed here only to make the experiment reproducible from independent image pairs.

In [ ]:
def make_features(hidden_states, last_hidden, raw, source):
    hs = {i: h for i, h in enumerate(hidden_states)}
    return BackboneOutput(
        cls=last_hidden[:, 0, :],
        hidden_states=hs,
        patch_tokens=last_hidden[:, 1:, :],
        metadata={
            "source": str(source),
            "model_name": "vits14",
            "hf_id": MODEL_ID,
            "num_frames": 1,
            "image_size": IMAGE_SIZE,
            "feature_dim": DIM,
            "num_patches": NUM_PATCHES,
            "num_layers": 13,
            "fps": 0.0,
            "stride": 1,
        },
        frames=raw,
    )

def make_action_features(cls_stack, source):
    return BackboneOutput(
        cls=cls_stack,
        hidden_states={},
        patch_tokens=None,
        metadata={
            "source": str(source),
            "model_name": "vits14",
            "hf_id": MODEL_ID,
            "num_frames": int(cls_stack.shape[0]),
            "image_size": IMAGE_SIZE,
            "feature_dim": DIM,
            "num_patches": NUM_PATCHES,
            "num_layers": 13,
            "fps": 0.0,
            "stride": 1,
        },
        frames=None,
    )

rows = []
camera_action = {}

for camera in CAMERAS:
    selected = pairs[camera][:PAIRS_PER_CAMERA]
    base_cls = []
    sel_cls_list = []

    print(f"\n===== {camera}: {len(selected)} pairs =====")

    # Warm up the downstream detector once.
    warm = selected[0]["current"]
    warm_out, warm_raw, _ = full_forward(warm)
    warm_feat = make_features(
        warm_out.hidden_states,
        warm_out.last_hidden_state,
        warm_raw,
        warm,
    )
    warm_det = detection_head.predict(warm_feat)
    assert warm_det.error is None, f"Detection smoke test failed: {warm_det.error}"

    for j, item in enumerate(selected):
        print(f"{camera}: {j+1}/{len(selected)} frame {item['frame_id']}")

        # Full current-frame baseline.
        full_out, full_raw, full_time = full_forward(item["current"])
        full_hidden = list(full_out.hidden_states)
        full_last = full_out.last_hidden_state.detach()
        base_cls.append(full_last[:, 0, :].detach())

        # Previous-frame cache. This is the state that would already exist
        # in a streaming deployment.
        prev_cache = build_previous_cache(item["previous"])

        # Lightweight change detector.
        mask, _ = image_patch_change_mask(
            item["current"],
            item["previous"],
        )

        # Selective current-frame representation.
        sel = selective_forward(
            item["current"],
            prev_cache,
            mask,
        )

        # Store CLS without retaining unnecessary computation graphs.
        sel_cls = sel["last_hidden"][:, 0, :].detach()
        sel_cls_list.append(sel_cls.cpu())

        # Detection on baseline features.
        base_feat = make_features(
            full_hidden,
            full_last,
            full_raw,
            item["current"],
        )

        # Detection on selective reconstructed features.
        sel_feat = make_features(
            sel["hidden_states"],
            sel["last_hidden"],
            sel["raw"],
            item["current"],
        )

        t_det0 = time.perf_counter()
        base_det = detection_head.predict(base_feat)
        base_det_time = time.perf_counter() - t_det0

        t_det1 = time.perf_counter()
        sel_det = detection_head.predict(sel_feat)
        sel_det_time = time.perf_counter() - t_det1

        assert base_det.error is None, f"Baseline detection failed: {base_det.error}"
        assert sel_det.error is None, f"Selective detection failed: {sel_det.error}"

        base_dets = base_det.primary[0] if base_det.primary else []
        sel_dets = sel_det.primary[0] if sel_det.primary else []

        det_cmp = compare_detections(base_dets, sel_dets)

        # Feature-level fidelity.
        a = full_last[:, 1:, :].float()
        b = sel["last_hidden"][:, 1:, :].float()
        cos = F.cosine_similarity(a, b, dim=-1)
        rel_l2 = torch.linalg.vector_norm(a - b, dim=-1) / (
            torch.linalg.vector_norm(a, dim=-1).clamp_min(1e-8)
        )

        rows.append({
            "camera": camera,
            "frame_id": int(item["frame_id"]),
            "mode": "temporal_reuse",
            "changed_percent": float(mask.mean() * 100.0),
            "reused_percent": float((~mask).mean() * 100.0),
            "selected_tokens": int(sel["selected_tokens"]),
            "token_fraction": float(sel["selected_tokens"] / 2026.0),
            "full_dinov2_seconds": float(full_time),
            "selective_backbone_seconds": float(sel["elapsed"]),
            "mean_patch_cosine": float(cos.mean().cpu()),
            "median_patch_cosine": float(cos.median().cpu()),
            "mean_relative_l2": float(rel_l2.mean().cpu()),
            "baseline_detection_seconds": float(base_det_time),
            "selective_detection_seconds": float(sel_det_time),
            **det_cmp,
        })

        # Release large tensors before the next frame.
        del full_out, prev_cache, sel, base_feat, sel_feat
        torch.cuda.empty_cache()

    camera_action[camera] = {
        "baseline_cls": torch.cat(base_cls, dim=0).cpu(),
        "selective_cls": torch.cat(sel_cls_list, dim=0),
    }

df = pd.DataFrame(rows)
df.to_csv(OUT / "stage4_frame_results.csv", index=False)

print("\nFrame-level evaluation complete.")
print("Rows:", len(df))
display(df.head())

## 10. Action recognition comparison

The professor's action head uses a trained Kinetics-400 linear probe over aggregated CLS tokens. We therefore feed the same 32-frame CLS sequence to the action head for the baseline and selective paths.

This tests whether the temporal-reuse representation changes the downstream action prediction.

In [ ]:
action_rows = []

for camera in CAMERAS:
    base_cls = camera_action[camera]["baseline_cls"]
    sel_cls = camera_action[camera]["selective_cls"]

    base_features = make_action_features(
        base_cls.to(DEVICE),
        f"stage4://{camera}/baseline",
    )
    sel_features = make_action_features(
        sel_cls.to(DEVICE),
        f"stage4://{camera}/temporal_reuse",
    )

    base_action = action_head.predict(base_features)
    sel_action = action_head.predict(sel_features)

    assert base_action.error is None, f"{camera} baseline action failed: {base_action.error}"
    assert sel_action.error is None, f"{camera} selective action failed: {sel_action.error}"

    base_top = base_action.details.get("top_k", [])
    sel_top = sel_action.details.get("top_k", [])

    base_primary = base_action.primary
    sel_primary = sel_action.primary

    action_rows.append({
        "camera": camera,
        "baseline_action": base_primary,
        "baseline_confidence": float(base_action.confidence),
        "selective_action": sel_primary,
        "selective_confidence": float(sel_action.confidence),
        "top1_agreement": bool(base_primary == sel_primary),
        "confidence_abs_diff": abs(
            float(base_action.confidence) - float(sel_action.confidence)
        ),
        "baseline_top5": json.dumps(base_top),
        "selective_top5": json.dumps(sel_top),
    })

action_df = pd.DataFrame(action_rows)
action_df.to_csv(OUT / "stage4_action_results.csv", index=False)

display(action_df[
    [
        "camera",
        "baseline_action",
        "baseline_confidence",
        "selective_action",
        "selective_confidence",
        "top1_agreement",
        "confidence_abs_diff",
    ]
])

## 11. Aggregate downstream results

The key metrics are:

### Detection
- baseline/selective detection counts
- baseline match rate
- selective match rate
- mean matched IoU
- confidence difference

### Action
- top-1 agreement
- confidence difference

### Backbone
- token reuse
- token fraction processed
- feature cosine similarity
- relative L2 error

Do **not** interpret token reuse as an achieved speedup.

In [ ]:
summary = (
    df.groupby("camera")
      .agg(
          frames=("frame_id", "count"),
          mean_reused_percent=("reused_percent", "mean"),
          mean_changed_percent=("changed_percent", "mean"),
          mean_selected_tokens=("selected_tokens", "mean"),
          mean_token_fraction=("token_fraction", "mean"),
          mean_full_backbone_time=("full_dinov2_seconds", "mean"),
          mean_selective_backbone_time=("selective_backbone_seconds", "mean"),
          mean_patch_cosine=("mean_patch_cosine", "mean"),
          mean_relative_l2=("mean_relative_l2", "mean"),
          baseline_detections=("baseline_dets", "sum"),
          selective_detections=("selective_dets", "sum"),
          matched_detections=("matched_dets", "sum"),
          mean_baseline_match_rate=("baseline_match_rate", "mean"),
          mean_selective_match_rate=("selective_match_rate", "mean"),
          mean_matched_iou=("mean_matched_iou", "mean"),
          mean_detection_conf_abs_diff=("mean_conf_abs_diff", "mean"),
      )
      .reset_index()
)

summary["theoretical_token_reduction_percent"] = (
    1.0 - summary["mean_token_fraction"]
) * 100.0

summary["backbone_time_ratio"] = (
    summary["mean_full_backbone_time"] /
    summary["mean_selective_backbone_time"]
)

action_summary = action_df.groupby("camera").agg(
    action_top1_agreement=("top1_agreement", "mean"),
    action_conf_abs_diff=("confidence_abs_diff", "mean"),
).reset_index()

summary = summary.merge(action_summary, on="camera", how="left")

summary.to_csv(OUT / "stage4_summary.csv", index=False)

display(summary)

In [ ]:
overall = {
    "mean_reused_percent": float(df["reused_percent"].mean()),
    "mean_changed_percent": float(df["changed_percent"].mean()),
    "mean_selected_tokens": float(df["selected_tokens"].mean()),
    "mean_token_fraction": float(df["token_fraction"].mean()),
    "theoretical_token_reduction_percent": float((1 - df["token_fraction"].mean()) * 100),
    "mean_patch_cosine": float(df["mean_patch_cosine"].mean()),
    "mean_relative_l2": float(df["mean_relative_l2"].mean()),
    "mean_baseline_detections": float(df["baseline_dets"].mean()),
    "mean_selective_detections": float(df["selective_dets"].mean()),
    "mean_baseline_match_rate": float(df["baseline_match_rate"].mean()),
    "mean_selective_match_rate": float(df["selective_match_rate"].mean()),
    "mean_matched_iou": float(df["mean_matched_iou"].mean()),
    "mean_detection_conf_abs_diff": float(df["mean_conf_abs_diff"].mean()),
    "mean_full_backbone_seconds": float(df["full_dinov2_seconds"].mean()),
    "mean_selective_backbone_seconds": float(df["selective_backbone_seconds"].mean()),
    "backbone_time_ratio": float(
        df["full_dinov2_seconds"].mean() / df["selective_backbone_seconds"].mean()
    ),
    "action_top1_agreement": float(action_df["top1_agreement"].mean()),
    "action_conf_abs_diff": float(action_df["confidence_abs_diff"].mean()),
}

print("STAGE 4 OVERALL")
print("=" * 70)
for k, v in overall.items():
    print(f"{k}: {v:.6f}" if isinstance(v, float) else f"{k}: {v}")

## 12. Interpretation guardrails

This experiment should be reported carefully.

A strong Stage 4 result would show:

- substantial reusable-token fraction,
- acceptable detection agreement with the full model,
- acceptable action top-1 agreement,
- and eventually a real implementation speedup after removing Python/cache overhead.

The current prototype may still be slower than full DINOv2 and may show representation drift. That is expected for an experimental reduced-token approximation and is **not** evidence that the research idea has failed.

The most important result is whether downstream perception remains stable enough to justify further engineering of a true token-level incremental transformer implementation.

In [ ]:
report = {
    "stage": "Stage 4 - Downstream Detection and Action Evaluation",
    "dataset": "WoodScape",
    "model": "DINOv2 ViT-S/14",
    "model_id": MODEL_ID,
    "resolution": "640x640",
    "patch_grid": "45x45",
    "patch_tokens": 2025,
    "embedding_dimension": 384,
    "cameras": CAMERAS,
    "pairs_per_camera": PAIRS_PER_CAMERA,
    "change_detector": "mean absolute RGB difference per 14x14 patch",
    "pixel_diff_threshold": PIXEL_DIFF_THRESHOLD,
    "stage2_reference_cosine": COSINE_REFERENCE,
    "downstream": {
        "detection": "Professor UnifiedPerception shared Adapter-v2 + YOLO26",
        "action": "Professor UnifiedPerception K400 linear-probe action head",
    },
    "outputs": [
        "stage4_frame_results.csv",
        "stage4_summary.csv",
        "stage4_action_results.csv",
        "stage4_summary.json",
    ],
    "scientific_note": (
        "The selective path is an approximate reduced-token implementation of "
        "global DINOv2 attention. Token reuse is not equivalent to achieved "
        "runtime speedup. Detection and action outputs are compared against "
        "the full DINOv2 baseline."
    ),
    "overall": overall,
}

with open(OUT / "stage4_summary.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

## 13. Create a downloadable ZIP

Run this only after all Stage 4 cells complete successfully.

In [ ]:
import shutil

zip_base = Path("/kaggle/working/stage4_downstream_32pairs")
zip_path = Path(shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUT,
))

print("ZIP created:", zip_path)
print(f"Size: {zip_path.stat().st_size / (1024**2):.2f} MB")